In [2]:
!pip install pycountry

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 37.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import os
import pycountry

DATA_PATH = "/content/drive/MyDrive/StackOverFlow/survey_results_public.csv"
OUT_DIR = "eda_images"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

ISO3_MANUAL = {
    "United States of America": "USA",
    "United Kingdom of Great Britain and Northern Ireland": "GBR",
    "Russian Federation": "RUS",
    "Republic of Korea": "KOR",
    "Democratic People's Republic of Korea": "PRK",
    "Iran, Islamic Republic of...": "IRN",
    "United Republic of Tanzania": "TZA",
    "Viet Nam": "VNM",
    "Venezuela, Bolivarian Republic of...": "VEN",
    "Bolivia, Plurinational State of...": "BOL",
    "Syrian Arab Republic": "SYR",
    "Taiwan": "TWN",
    "Congo, Republic of the...": "COG",
    "Lao People's Democratic Republic": "LAO",
    "Czech Republic": "CZE",
    "Hong Kong (S.A.R.)": "HKG",
    "Brunei Darussalam": "BRN",
    "Libyan Arab Jamahiriya": "LBY",
    "Palestine": "PSE",
    "South Korea": "KOR",
    "Iran": "IRN",
    "Russia": "RUS",
    "UK": "GBR",
    "Vietnam": "VNM"
}

def to_iso3(name):
    if pd.isna(name): return None
    if name in ISO3_MANUAL: return ISO3_MANUAL[name]
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None

if "ConvertedCompYearly" in df.columns:
    sal = pd.to_numeric(df["ConvertedCompYearly"], errors="coerce")
    df["_sal"] = sal
    q_low, q_high = sal.quantile(0.05), sal.quantile(0.95)
    df["salary_filtered"] = np.where(df["_sal"].between(q_low, q_high), df["_sal"], np.nan)

if "WorkExp" in df.columns:
    df["_wexp"] = pd.to_numeric(df["WorkExp"], errors="coerce")
    bins = [0, 1, 3, 5, 10, 15, 20, 30, 100]
    labels = ["<1", "1-3", "3-5", "5-10", "10-15", "15-20", "20-30", "30+"]
    df["wexp_bin"] = pd.cut(df["_wexp"], bins=bins, labels=labels, right=False)

if "Country" in df.columns and "ConvertedCompYearly" in df.columns:
    cs = df[df["_sal"].notna()].groupby("Country")["_sal"].agg(["median", "mean", "count"])
    cs.columns = ["median_salary", "mean_salary", "respondents"]
    cs_map = cs.reset_index()
    cs_map["country_short"] = cs_map["Country"]
    cs_map["iso3"] = cs_map["Country"].apply(to_iso3)
    cs_map_out = cs_map[["Country", "country_short", "iso3", "median_salary", "mean_salary", "respondents"]]
    cs_map_out.to_csv(os.path.join(OUT_DIR, "04b_salary_by_country_for_map.csv"), index=False)

df.to_csv("survey_cleaned_base.csv", index=False)